# Effect Size & Practical Significance
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import FloatSlider, IntSlider, Output, VBox

from IPython.display import display, clear_output
from scipy import stats

from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_slider, make_int_slider, display_widget, register_observer

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** Cohen's d as a standardized measure of separation between two groups
- **Explain** why statistical significance and practical significance are not the same thing
- **Interpret** effect size benchmarks (small, medium, large) in a real decision-making context

> **Tip:** First raise the sample size to 10,000 and set the mean difference to 0.5 — you will get a very significant p-value (p < 0.001) despite a tiny Cohen's d. Then lower the sample to 30 with a mean difference of 5.0 — often not significant despite a massive effect.

---
## How we got here

In *17–20* we learned hypothesis testing: p-values tell us whether an observed difference is likely to be real or just sampling noise. But p-values are silent on a different question: even if the difference is real, is it *big enough to matter*? Effect size answers that question.

---
## Why this matters for data science

In machine learning you run comparisons constantly: model A vs model B, feature set X vs feature set Y, hyperparameter setting 1 vs 2. With large datasets, nearly any difference will reach statistical significance. Effect size keeps you from celebrating improvements that are real but too small to matter in production. It also rescues meaningful differences found in small datasets that a p-value threshold alone would reject.

---
## Try it yourself

In [2]:
out = make_output()

diff_slider = make_slider(0.0, 8.0, 0.1, 2.0, "Mean difference:")
n_slider    = make_int_slider(10, 5000, 10, 50, "Sample size (n):")

SIGMA = 4.0  # pooled std dev (fixed)
x_range = np.linspace(-20, 30, 600)

COHEN_LABELS = [
    (0.20, "Small (d = 0.2)"),
    (0.50, "Medium (d = 0.5)"),
    (0.80, "Large (d = 0.8)"),
]


def render(diff, n):
    with out:
        clear_output(wait=True)

        mu_a, mu_b = 10.0, 10.0 + diff
        cohens_d = diff / SIGMA

        se = SIGMA * np.sqrt(2 / n)
        t_stat = diff / se if se > 0 else 0
        p_val  = 2 * stats.t.sf(abs(t_stat), df=2 * n - 2)

        sig_label = "p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
        sig_star  = "Statistically significant" if p_val < 0.05 else "NOT significant"

        if cohens_d < 0.20:
            d_label = "Negligible"
        elif cohens_d < 0.50:
            d_label = "Small"
        elif cohens_d < 0.80:
            d_label = "Medium"
        else:
            d_label = "Large"

        pdf_a = stats.norm.pdf(x_range, mu_a, SIGMA)
        pdf_b = stats.norm.pdf(x_range, mu_b, SIGMA)

        # Overlap area: shaded region where both PDFs have probability
        overlap = np.minimum(pdf_a, pdf_b)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([pdf_a, np.zeros_like(pdf_a)]),
            fill="toself",
            fillcolor="rgba(76,110,245,0.20)",
            line=dict(color=PALETTE["primary"], width=2),
            name=f"Group A  (μ = {mu_a:.1f})",
        ))
        fig.add_trace(go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([pdf_b, np.zeros_like(pdf_b)]),
            fill="toself",
            fillcolor="rgba(247,103,7,0.20)",
            line=dict(color=PALETTE["secondary"], width=2),
            name=f"Group B  (μ = {mu_b:.1f})",
        ))
        fig.add_trace(go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([overlap, np.zeros_like(overlap)]),
            fill="toself",
            fillcolor="rgba(150,100,200,0.25)",
            line=dict(color="rgba(150,100,200,0.60)", width=1, dash="dot"),
            name="Overlap",
        ))

        layout = base_layout(
            title=(
                f"Cohen's d = {cohens_d:.2f} ({d_label})   |   "
                f"{sig_label}   [{sig_star}]   |   n = {n} per group"
            ),
            xaxis_title="Outcome value",
            yaxis_title="Probability Density",
        )
        layout.update(showlegend=True)
        fig.update_layout(layout)
        display(go.FigureWidget(fig))


register_observer(
    [diff_slider, n_slider],
    lambda change: render(diff_slider.value, n_slider.value)
)

display_widget(VBox([diff_slider, n_slider]), out)
render(diff_slider.value, n_slider.value)

---
## What's happening?

**Cohen's d** standardizes a mean difference by the pooled standard deviation, making it comparable across studies regardless of scale:

$$d = \frac{\mu_A - \mu_B}{\sigma_{\text{pooled}}}$$

It asks: how many standard deviations apart are the two group means?

| Cohen's d | Conventional label | Rough meaning |
|-----------|-------------------|---------------|
| 0.0–0.2 | Negligible | Groups overlap almost completely |
| 0.2–0.5 | Small | Noticeable with a careful look |
| 0.5–0.8 | Medium | Clearly visible in a chart |
| > 0.8 | Large | Groups are substantially separated |

### The p-value trap

The p-value is affected by both **effect size** and **sample size**. Holding effect size fixed:

$$t = \frac{d}{\sqrt{2/n}} \quad \Rightarrow \quad \text{larger } n \Rightarrow \text{smaller } p$$

This means:

- With **large n**: even trivially small effects reach p < 0.001
- With **small n**: even large effects may not reach significance

**Statistical significance** answers: is this difference real, or is it sampling noise?  
**Practical significance** answers: is this difference big enough to matter?

Both questions matter. Report effect size alongside p-values.

### r² as effect size for regression

For regression models, the equivalent measure is r² (the coefficient of determination):

$$r^2 = 1 - \frac{\text{SS}_{\text{residual}}}{\text{SS}_{\text{total}}}$$

It answers: what fraction of the variance in y does the model explain? An r² of 0.03 may be statistically significant but practically useless. An r² of 0.65 explains most of the variance and is likely actionable.

---
## Real-world example: A/B test for a homepage redesign

A company runs an A/B test of a new homepage. Version B increases average session duration by 4 seconds. With 50,000 users this is highly significant (p < 0.0001). Should they ship it?

The chart below simulates both conditions and shows why a statistically overwhelming result can mask a practically negligible difference.

Notice:

- The p-value is essentially zero because the massive sample size amplifies any real difference
- The distributions are nearly indistinguishable, Cohen's d is tiny
- A 4-second gain against a mean session of 180 seconds is a 2.2% improvement

> **Discussion question:** What additional business context would you need before deciding whether a Cohen's d of 0.05 justifies the cost of shipping the new homepage?

In [3]:
np.random.seed(42)
n_ab    = 50_000
mu_ctrl = 180.0
mu_test = 184.0
sigma   = 80.0

ctrl = np.random.normal(mu_ctrl, sigma, n_ab)
test = np.random.normal(mu_test, sigma, n_ab)

t_stat, p_val = stats.ttest_ind(test, ctrl)
cohens_d      = (np.mean(test) - np.mean(ctrl)) / np.sqrt(
    (np.std(ctrl) ** 2 + np.std(test) ** 2) / 2
)

x_ab = np.linspace(0, 500, 600)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=ctrl, histnorm="probability density", nbinsx=60,
    marker_color=PALETTE["primary"], opacity=0.45,
    name=f"Control  μ={np.mean(ctrl):.1f}s"
))
fig.add_trace(go.Histogram(
    x=test, histnorm="probability density", nbinsx=60,
    marker_color=PALETTE["secondary"], opacity=0.45,
    name=f"Variant  μ={np.mean(test):.1f}s"
))

sig_label = "p < 0.0001" if p_val < 0.0001 else f"p = {p_val:.4f}"
layout = base_layout(
    title=(
        f"A/B Test: Session Duration   |   "
        f"{sig_label} (highly significant!)   |   Cohen's d = {cohens_d:.3f} (negligible)"
    ),
    xaxis_title="Session duration (seconds)",
    yaxis_title="Probability Density",
)
layout.update(barmode="overlay", showlegend=True)
fig.update_layout(layout)
fig.show()

### Effect size measures across common ML scenarios

| Comparison | Effect size measure | How to interpret |
|------------|--------------------|-----------------|
| Model A vs Model B accuracy | Cohen's d on cross-val scores | Is the improvement worth the added complexity? |
| Feature engineering gain in r² | Δr² = r²_new − r²_old | Does adding the feature explain meaningfully more variance? |
| Class imbalance effect on recall | Absolute recall difference | A 2pp recall gain at 95% base is different from 2pp at 50% |
| A/B test conversion rate | Cohen's h (for proportions) | Is the conversion lift big enough to justify rollout cost? |
| Fairness gap between subgroups | Cohen's d on outcome rates | Quantifies disparity independently of sample size |

---
## Key takeaway

> **Statistical significance tells you whether a difference is real; effect size tells you whether it is worth caring about — and with large datasets, nearly everything is significant, so effect size becomes the more important question.**

---
*Next up: Supervised learning — applying these statistical foundations to build models that learn from labeled data*